<a href="https://colab.research.google.com/github/induwaralakshan4/AI-Generated-Image-Recognition/blob/main/App.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import sklearn
print(sklearn.__version__)
# request sklearn version 1.6.1

1.6.1


In [3]:
import joblib
model=joblib.load('Ai_and_Real_Face_Recognize.sav')

In [17]:
import gradio as gr
import numpy as np
import cv2
from PIL import Image
import joblib
model=joblib.load('Ai_and_Real_Face_Recognize.sav')

# Model labels
label_dict = {
    0: "Real Images",
    1: "AI-Generated Images"
}

# Real face examples
real_images = [
    "real_faces/1.jpg",
    "real_faces/2.jpg",
    "real_faces/3.jpg"
]

# AI-generated face examples
ai_images = [
    "ai_faces/1.jpg",
    "ai_faces/2.png",
    "ai_faces/3.png"
]


# Image preprocessing and prediction
def process_image(image):
    if image is None:
        return "Please upload or select an image first!"

    try:
        # Convert input to a valid RGB NumPy array
        if isinstance(image, str):
            image = Image.open(image).convert("RGB")
            img_array = np.array(image)

        elif isinstance(image, Image.Image):
            img_array = np.array(image.convert("RGB"))

        else:
            img_array = np.asarray(image)

        if img_array.dtype == object:
            return "Invalid image format!"

        # Convert to grayscale
        if img_array.ndim == 2:
            gray = img_array

        else:
            if img_array.shape[-1] == 4:
                img_array = cv2.cvtColor(
                    img_array, cv2.COLOR_RGBA2RGB
                )

            gray = cv2.cvtColor(
                img_array, cv2.COLOR_RGB2GRAY
            )

        # Resize to 8x8 pixels
        gray = cv2.resize(gray, (8, 8))

        # Prepare model input
        features = gray.reshape(1, 64)

        # Predict
        result = model.predict(features)[0]

        return label_dict.get(
            int(result), f"Unknown class: {result}"
        )

    except Exception as e:
        return f"Prediction error: {e}"


# Select a gallery example using its index
def select_real_example(evt: gr.SelectData):
    index = evt.index

    if isinstance(index, (tuple, list)):
        index = index[0]

    image_path = real_images[int(index)]

    image = np.array(
        Image.open(image_path).convert("RGB")
    )

    prediction = process_image(image)

    return image, prediction


def select_ai_example(evt: gr.SelectData):
    index = evt.index

    if isinstance(index, (tuple, list)):
        index = index[0]

    image_path = ai_images[int(index)]

    image = np.array(
        Image.open(image_path).convert("RGB")
    )

    prediction = process_image(image)

    return image, prediction


# Gradio interface
with gr.Blocks() as demo:

    gr.Markdown("# AI-Generated Image Recognition")

    gr.Markdown(
        "Upload your own image or click an example image "
        "to automatically predict its category."
    )

    with gr.Row():
        with gr.Column():

            image_input = gr.Image(
                label="Upload Your Image",
                type="numpy"
            )

            submit_btn = gr.Button(
                "Predict",
                variant="primary"
            )


        with gr.Column():

            output_text = gr.Textbox(
                label="Prediction Result",
                interactive=False
            )
            gr.Markdown("## Example Image")

            gr.Markdown("### Real Face Images")

            real_gallery = gr.Gallery(
                value=real_images,
                label="Click a Real Face Image",
                columns=3,
                height=200,
                object_fit="contain"
            )

            gr.Markdown("### AI-Generated Face Images")

            ai_gallery = gr.Gallery(
                value=ai_images,
                label="Click an AI-Generated Face",
                columns=3,
                height=200,
                object_fit="contain"
            )




    submit_btn.click(
        fn=process_image,
        inputs=image_input,
        outputs=output_text
    )


    real_gallery.select(
        fn=select_real_example,
        inputs=[],
        outputs=[image_input, output_text]
    )


    ai_gallery.select(
        fn=select_ai_example,
        inputs=[],
        outputs=[image_input, output_text]
    )



if __name__ == "__main__":
    demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5e805b153b6e3b5d56.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
